# CAMEL Model for all the cooperatives

## Functions

In [1]:
import pandas as pd

### Capital Adequacy

In [2]:
def get_value(excel_file, cuenta, company_name, year, month):

    year = int(year)
    month = int(month)

    mask = (
        (excel_file['CUENTA'] == cuenta) &
        (excel_file['YEAR'] == year) &
        (excel_file['MONTH'] == month)
    )

    series = excel_file.loc[mask, company_name]

    if series.empty or series.isna().all():
        return None

    return series.iloc[-1]  # último corte del mes, no el primero

# Loss of Assets
def loss_of_assets(excel_file, company_name, year, month):

    v1 = get_value(
        excel_file, 300000, company_name, year, month
    )

    v2 = get_value(
        excel_file, 310000, company_name, year, month
    )

    if v1 is None or v2 is None or v2 == 0:
        return None

    return v1 / v2

# Solvency Ratio
def solvency_ratio():
    return 0

# Indicator of the relationship between the non-reducible minimum social contributions and the Share Capital
def min_social_contrib_social_capital(
    excel_file, company_name, year, month
):

    value1 = get_value(
        excel_file, 311000, company_name, year, month
    )

    value2 = get_value(
        excel_file, 310000, company_name, year, month
    )

    if value1 is None or value2 is None:
        return 0

    return value1 / value2 if value2 != 0 else 0


# --- Sum the values of every row 'i' in a specific company
def summation(excel_file, index_list, company_name, year, month):

    suma = 0

    year = int(year)
    month = int(month)

    for i in index_list:

        mask = (
            (excel_file['CUENTA'] == i) &
            (excel_file['YEAR'] == year) &
            (excel_file['MONTH'] == month)
        )

        account_coop = excel_file.loc[mask, company_name]

        if account_coop.empty or account_coop.isna().all():
            value = 0
        else:
            value = account_coop.iloc[-1]

        suma += value

    return suma


# Indicator of the relationship between Institutional Capital and Total Assets
def capital_contribution_ratio(
    excel_file, company_name, year, month
):

    capital_inst_indices = [320000, 330000, 340000]

    capital_inst = summation(
        excel_file,
        capital_inst_indices,
        company_name,
        year,
        month
    )

    total_assets = get_value(
        excel_file, 100000, company_name, year, month
    )

    if total_assets is None:
        return 0

    return (
        capital_inst / total_assets
        if total_assets != 0
        else 0
    )

### Assets Quality

In [3]:
# Functions for quality indicator by risk with penalties
def gross_portfolio(excel_file, company_name, year, month):
    gross_portfolio_indices = [
        140400, 140500, 141100, 141200, 144100, 144200,
        144800, 145400, 145500, 146100, 146200, 146900
    ]
    return summation(excel_file, gross_portfolio_indices, company_name, year, month)

def risk_quality_indicator(excel_file, company_name, year, month):
    risk_portfolio_indices = [
        140410, 140415, 140420, 140425, 140510, 140515, 140520, 140525,
        141110, 141115, 141120, 141125, 141210, 141215, 141220, 141225,
        144110, 144115, 144120, 144125, 144210, 144215, 144220, 144225,
        144810, 144815, 144820, 144825, 145410, 145415, 145420, 145425,
        145510, 145515, 145520, 145525, 146110, 146115, 146120, 146125,
        146210, 146215, 146220, 146225, 146910, 146915, 146920, 146925,
        146935, 146940, 146945, 146950
    ]
    qualified_portfolio = summation(excel_file, risk_portfolio_indices, company_name, year, month)
    gross_portf = gross_portfolio(excel_file,company_name, year, month)

    return qualified_portfolio / gross_portf if gross_portf != 0 else 0

def risk_quality_with_writeoffs(excel_file, company_name, year, month):
    # total qualified portfolio
    qualified_portfolio = risk_quality_indicator(excel_file, company_name, year, month)
    # writeoffs
    mask = (
            (excel_file['CUENTA'] == 831015) &
            (excel_file['YEAR'] == year) &
            (excel_file['MONTH'] == month)
        )
    
    writeoffs_list = list(excel_file.loc[mask, company_name])
    writeoffs = writeoffs_list[0] if writeoffs_list else 0
    total_with_writeoffs_indices = [
        140400, 140500, 141100, 141200, 144100, 144200, 144800,
        145400, 145500, 146100, 146200, 146900, 831015
    ]
    total_with_writeoffs = summation(excel_file, total_with_writeoffs_indices, company_name, year, month)

    return (qualified_portfolio + writeoffs) / total_with_writeoffs if total_with_writeoffs != 0 else 0

# Total Portfolio at Risk Coverage Indicator
def total_risk_coverage_indicator(excel_file, company_name, year, month):
    deterioration_indices = [140800, 144500, 145100, 145800, 146500, 146800, 147100]
    provisions = summation(excel_file, deterioration_indices, company_name, year, month)
    gross_portf = gross_portfolio(excel_file, company_name, year, month)
    return provisions / gross_portf if gross_portf != 0 else 0

# Productive asset
def productive_assets_ratio(excel_file, company_name, year, month):
    productive_assets_indices = [
        112000, 120000, 130000, 140405, 140410, 140505, 140510,
        141105, 141110, 141205, 141210, 144105, 144110, 144205,
        144210, 144805, 144810, 145405, 145410, 145505, 145510,
        146105, 146110, 146205, 146210, 146905, 146910, 146930,
        146935, 160505, 161505
    ]
    productive_assets = summation(excel_file, productive_assets_indices, company_name, year, month)
    mask = (
                (excel_file['CUENTA'] == 100000) &
                (excel_file['YEAR'] == year) &
                (excel_file['MONTH'] == month)
            )
    total_assets_list = list(excel_file.loc[mask, company_name])
    total_assets = total_assets_list[0] if total_assets_list else 0
    return productive_assets / total_assets if total_assets != 0 else 0

# Individual Coverage Indicator of the Unproductive Portfolio for the At-Risk Portfolio
def individual_coverage_nonproductive_portfolio(excel_file, company_name, year, month):
    provisions_cde_indices = [
        140815, 140820, 140825,
        144525, 144530, 144535, 144540, 144545, 144550,
        145115, 145120, 145125,
        145825, 145830, 145835, 145840, 145845, 145850,
        146525, 146530, 146535, 146540, 146545, 146550,
        147115, 147120, 147125, 147140, 147145, 147150
    ]
    provisions = summation(excel_file, provisions_cde_indices, company_name, year, month)

    overdue_portfolio_indices = [
        140415, 140420, 140425, 140515, 140520, 140525,
        141115, 141120, 141125, 141215, 141220, 141225,
        144115, 144120, 144125, 144215, 144220, 144225,
        144815, 144820, 144825, 145415, 145420, 145425,
        145515, 145520, 145525, 146115, 146120, 146125,
        146215, 146220, 146225, 146915, 146920, 146925,
        146940, 146945, 146950
    ]
    overdue = summation(excel_file, overdue_portfolio_indices, company_name, year, month)

    return provisions / overdue if overdue != 0 else 0

### Managerial Quality

In [4]:
# Operating Financial Margin Indicator
def financial_margin_operation(excel_file, company_name, year, month):
    pos_list = list(excel_file.loc[(excel_file['CUENTA'] == 410000) & (excel_file['YEAR'] == year) & (excel_file['MONTH'] == month), company_name])
    positive_margin = pos_list[0] if pos_list else 0

    neg_list1 = list(excel_file.loc[(excel_file['CUENTA'] == 610000) & (excel_file['YEAR'] == year) & (excel_file['MONTH'] == month), company_name])
    negative_margin1 = neg_list1[0] if neg_list1 else 0

    serie = excel_file.loc[(excel_file['CUENTA'] == 700000) & (excel_file['YEAR'] == year) & (excel_file['MONTH'] == month), company_name]
    negative_margin2 = 0 if serie.empty else serie.iloc[0]

    sales_list = list(excel_file.loc[(excel_file['CUENTA'] == 410000) & (excel_file['YEAR'] == year) & (excel_file['MONTH'] == month), company_name])
    sales_income = sales_list[0] if sales_list else 0

    negative_margin = negative_margin1 + negative_margin2
    return (positive_margin - negative_margin) / sales_income if sales_income != 0 else 0


# Operating Margin Indicator (usa 'summation', no se modifica)
def operational_margin(excel_file, company_name, year, month):
    income_pos_indices = [410000, 422500]
    income_neg_indices = [610000, 700000, 510500, 510700, 511000, 511500, 540000]
    sales_indices = [410000, 422500]
    income_pos = summation(excel_file, income_pos_indices, company_name, year, month)
    income_neg = summation(excel_file, income_neg_indices, company_name, year, month)
    sales_income = summation(excel_file, sales_indices, company_name, year, month)
    return (income_pos - income_neg) / sales_income if sales_income != 0 else 0

# Indicator of the relationship between financial obligations and total liabilities
def financial_obligations_ratio(excel_file, company_name, year, month):

    obligations_list = list(excel_file.loc[(excel_file['CUENTA'] == 230000) & (excel_file['YEAR'] == year) & (excel_file['MONTH'] == month), company_name])
    obligations = obligations_list[0] if obligations_list else 0

    total_liabilities_list = list(excel_file.loc[(excel_file['CUENTA'] == 200000) & (excel_file['YEAR'] == year) & (excel_file['MONTH'] == month), company_name])
    total_liabilities = total_liabilities_list[0] if total_liabilities_list else 0

    return obligations / total_liabilities if total_liabilities != 0 else 0

# Balance structure (usa 'summation', no se modifica)
def balance_structure(excel_file, company_name, year, month):
    productive_assets_indices = [
        112000, 120000, 130000, 140405, 140410, 140505, 140510,
        141105, 141110, 141205, 141210, 144105, 144110, 144205,
        144210, 144805, 144810, 145405, 145410, 145505, 145510,
        146105, 146110, 146205, 146210, 146905, 146910, 146930,
        146935, 160505, 161505
    ]
    interest_liabilities_indices = [210000, 230000]
    productive_assets = summation(excel_file, productive_assets_indices, company_name, year, month)
    interest_liabilities = summation(excel_file, interest_liabilities_indices, company_name, year, month)
    return productive_assets / interest_liabilities if interest_liabilities != 0 else 0


### Earnings Strength

In [13]:
# Net margin indicator
def net_margin_indicator(excel_file, company_name, year, month):
    net_surplus_list = list(excel_file.loc[(excel_file['CUENTA'] == 530000) & (excel_file['YEAR'] == int(year)) & (excel_file['MONTH'] == int(month)), company_name])
    net_surplus = net_surplus_list[0] if net_surplus_list else 0
    incomes_indices = [410000, 422500]
    total_incomes = summation(excel_file, incomes_indices, company_name, year, month)
    return net_surplus / total_incomes if total_incomes != 0 else 0

# Return on equity indicator
def ROE(excel_file, company_name, year, month, n=1):

    DATE_actual = pd.Period(f"{year}-{month:02d}", freq="M")
    meses_previos = [(DATE_actual - i) for i in range(1, 13)]

    num_list = list(excel_file.loc[
        (excel_file['CUENTA'] == 530000) &
        (excel_file['YEAR'] == year) &
        (excel_file['MONTH'] == month),
        company_name
    ])
    num = num_list[0] if num_list else 0

    # Máscara vectorizada de meses previos (mismo estilo que las funciones que sí funcionan)
    mask_previos = False
    for p in meses_previos:
        mask_previos = mask_previos | ((excel_file['YEAR'] == p.year) & (excel_file['MONTH'] == p.month))

    valores = excel_file.loc[
        (excel_file['CUENTA'] == 300000) & mask_previos,
        company_name
    ].values

    if len(valores) == 0 or n == 0 or num == 0:
        return None

    prom_300000 = valores.mean()

    if prom_300000 == 0:         
        return None
    
    roe = (1 + ((num / prom_300000) / n))**12 - 1

    return roe


# Indicator of return on invested capital
def ROIC(excel_file, company_name, year, month, n=1):

    DATE_actual = pd.Period(f"{year}-{month:02d}", freq="M")
    meses_previos = [(DATE_actual - i) for i in range(1, 13)]

    # Numerator
    num_list = list(excel_file.loc[
        (excel_file['CUENTA'] == 530000) &
        (excel_file['YEAR'] == year) &
        (excel_file['MONTH'] == month),
        company_name
    ])
    num = num_list[0] if num_list else 0

    # Máscara vectorizada de meses previos
    mask_previos = False
    for p in meses_previos:
        mask_previos = mask_previos | ((excel_file['YEAR'] == p.year) & (excel_file['MONTH'] == p.month))

    # Denominator
    cuentas = [210000, 230000, 300000]
    promedios = []

    for cuenta in cuentas:
        valores = excel_file.loc[
            (excel_file['CUENTA'] == cuenta) & mask_previos,
            company_name
        ].values
        prom = valores.mean() if len(valores) > 0 else 0
        promedios.append(prom)

    denom = sum(promedios)

    if denom == 0 or n == 0 or num == 0:
        return None  # division by zero or no data

    roic = (1 + ((num / denom) / n))**12 - 1

    return roic

### Liquidicy Efficiency

**Indicador de relación entre Activos Liquidos ampliados a depósitos de corto plazo (propuesto dentro de los indicadores de la SES)**

Indicador de relación entre Activos Liquidos ampliados a depósitos de corto plazo

= Activos líquidos ampliados / Depósitos Corto Plazo

= (Efectivo y equivalentess + Fondo Liquidez + Inversiones) / (Depósito < 6 Meses)

= $\frac{(C110500 + C111000 + C111500 + C112001 + C112003 + C112005 + C112006 + C112007 + C112008 + C120305 + C120310 + C120315 + C120320 + C120330+ C123016 + C120400 + C120800 + C121300 + C122800 + C123000)}{(C210500 + C211005 + C212505 + C212510 + C213005)}$

In [6]:
# Liquidity Risk Indicator
def liquidity_risk(coops, company, y, m):
    # Numerator: activos líquidos ampliados
    numerator_indices = [
        110500, 111000, 111500, 112001, 112003, 112005, 112006, 112007,
        112008, 120305, 120310, 120315, 120320, 120330, 123016, 120400,
        120800, 121300, 122800, 123000
    ]

    # Denominator: depósitos corto plazo
    denominator_indices = [210500, 211005, 212505, 212510, 213005]

    num = summation(coops, numerator_indices, company, y, m)
    den = summation(coops, denominator_indices, company, y, m)

    return num / den if den != 0 else 0

## Read the csv for all the cooperatives

In [7]:
coops = pd.read_csv("../../tablas/Datos_2022_2025_cooperativas_CACs.csv")
coops

,COOPERATIVA DE EMPLEADOS DE CAFAM,COOPERATIVA DE TRABAJADORES DE LA INDUSTRIA MILITAR,COOPERATIVA DEL SISTEMA NACIONAL DE JUSTICIA,COOPERATIVA DE LOS PROFESIONALES DE LA SALUD COASMEDAS,COOPERATIVA DE AHORRO Y CREDITO PARA EL BIENESTAR SOCIAL,COOPERATIVA PARA EL BIENESTAR SOCIAL,COOPERATIVA FINANCIERA SAN FRANCISCO,COOPERATIVA MULTIACTIVA DE LA AVIACION CIVIL COLOMBIANA,COOPERATIVA DE EMPLEADOS DE DOW COLOMBIA,PROGRESSA ENTIDAD COOPERATIVA DE AHORRO Y CREDITO,...,MICROEMPRESAS DE COLOMBIA COOPERATIVA DE AHORRO Y CREDITO,COOPERATIVA DE AHORRO Y CREDITO CAJA UNION,COOPERATIVA ESPECIALIZADA DE AHORRO Y CREDITO AFROAMERICANA,COOPERATIVA ESPECIALIZADA DE AHORRO Y CREDITO CANAPRO,LA COOPERATIVA DE AHORRO Y CREDITO SUCREDITO,COOPERTAIVA ESPECIALIZADA DE AHORRO Y CREDITO TAX LA FERIA,COOPERATIVA DE AHORRO Y CREDITO SUYA LTDA,CUENTA,NOMBRE CUENTA,DATE
0,1.453604e+11,1.204461e+10,0.0,3.387150e+11,1.560676e+11,9.879448e+10,1.064674e+10,7.021719e+10,3.751248e+10,2.452617e+11,...,1.964690e+11,0.0,7.498950e+09,0.0,7.673873e+10,3.689148e+10,0.0,100000,ACTIVO,2022-12-01
1,1.094558e+10,1.108710e+09,0.0,1.566943e+10,2.669495e+10,1.918907e+09,1.552585e+09,4.685907e+09,1.926925e+09,7.286577e+09,...,1.904275e+10,0.0,5.507006e+08,0.0,2.782317e+09,5.564935e+09,0.0,110000,EFECTIVO Y EQUIVALENTE AL EFECTIVO,2022-12-01
2,2.868046e+08,1.061263e+07,0.0,8.229352e+08,1.100000e+06,3.589941e+08,1.718631e+08,1.241549e+08,8.000000e+05,2.090386e+08,...,2.193096e+09,0.0,4.744467e+08,0.0,2.018711e+08,1.615997e+08,0.0,110500,CAJA,2022-12-01
3,2.791246e+08,1.061263e+07,0.0,8.180852e+08,0.000000e+00,3.589941e+08,1.718631e+08,1.146549e+08,0.000000e+00,2.080386e+08,...,2.182646e+09,0.0,4.738467e+08,0.0,1.998711e+08,1.600997e+08,0.0,110505,CAJA GENERAL,2022-12-01
4,7.680000e+06,0.000000e+00,0.0,4.850000e+06,1.100000e+06,0.000000e+00,0.000000e+00,9.500000e+06,8.000000e+05,1.000000e+06,...,1.045000e+07,0.0,6.000000e+05,0.0,2.000000e+06,1.500000e+06,0.0,110510,CAJA MENOR,2022-12-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112993,0.000000e+00,0.000000e+00,0.0,2.470562e+10,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.0,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.0,931500,BIENES Y VALORES RECIBIDOS EN ADMINISTRACIÓN,2022-09-01
112994,0.000000e+00,0.000000e+00,0.0,2.470562e+10,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.0,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.0,931505,CARTERA FOGACOOP,2022-09-01
112995,7.498403e+09,1.029522e+10,0.0,4.308271e+11,2.173890e+11,1.432583e+11,6.690272e+09,6.667182e+10,2.998245e+10,4.281462e+11,...,1.410674e+11,0.0,6.967973e+09,0.0,7.480500e+08,0.000000e+00,0.0,960000,ACREEDORAS POR CONTRA (DB),2022-09-01
112996,7.498403e+09,1.029522e+10,0.0,4.308271e+11,2.173890e+11,1.432583e+11,6.690272e+09,6.667182e+10,2.998245e+10,4.281462e+11,...,1.410674e+11,0.0,6.967973e+09,0.0,7.480500e+08,0.000000e+00,0.0,960500,RESPONSABILIDADES CONTINGENTES POR EL CONTRARIO,2022-09-01


In [8]:
coops["DATE"] = pd.to_datetime(coops["DATE"])
coops["YEAR"] = coops["DATE"].dt.year
coops["MONTH"] = coops["DATE"].dt.month
coops.head()

C:\Users\Liana\AppData\Local\Temp\ipykernel_7848\3537256573.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  coops["YEAR"] = coops["DATE"].dt.year
C:\Users\Liana\AppData\Local\Temp\ipykernel_7848\3537256573.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  coops["MONTH"] = coops["DATE"].dt.month


,COOPERATIVA DE EMPLEADOS DE CAFAM,COOPERATIVA DE TRABAJADORES DE LA INDUSTRIA MILITAR,COOPERATIVA DEL SISTEMA NACIONAL DE JUSTICIA,COOPERATIVA DE LOS PROFESIONALES DE LA SALUD COASMEDAS,COOPERATIVA DE AHORRO Y CREDITO PARA EL BIENESTAR SOCIAL,COOPERATIVA PARA EL BIENESTAR SOCIAL,COOPERATIVA FINANCIERA SAN FRANCISCO,COOPERATIVA MULTIACTIVA DE LA AVIACION CIVIL COLOMBIANA,COOPERATIVA DE EMPLEADOS DE DOW COLOMBIA,PROGRESSA ENTIDAD COOPERATIVA DE AHORRO Y CREDITO,...,COOPERATIVA ESPECIALIZADA DE AHORRO Y CREDITO AFROAMERICANA,COOPERATIVA ESPECIALIZADA DE AHORRO Y CREDITO CANAPRO,LA COOPERATIVA DE AHORRO Y CREDITO SUCREDITO,COOPERTAIVA ESPECIALIZADA DE AHORRO Y CREDITO TAX LA FERIA,COOPERATIVA DE AHORRO Y CREDITO SUYA LTDA,CUENTA,NOMBRE CUENTA,DATE,YEAR,MONTH
0,1.453604e+11,1.204461e+10,0.0,3.387150e+11,1.560676e+11,9.879448e+10,1.064674e+10,7.021719e+10,3.751248e+10,2.452617e+11,...,7.498950e+09,0.0,7.673873e+10,3.689148e+10,0.0,100000,ACTIVO,2022-12-01,2022,12
1,1.094558e+10,1.108710e+09,0.0,1.566943e+10,2.669495e+10,1.918907e+09,1.552585e+09,4.685907e+09,1.926925e+09,7.286577e+09,...,5.507006e+08,0.0,2.782317e+09,5.564935e+09,0.0,110000,EFECTIVO Y EQUIVALENTE AL EFECTIVO,2022-12-01,2022,12
2,2.868046e+08,1.061263e+07,0.0,8.229352e+08,1.100000e+06,3.589941e+08,1.718631e+08,1.241549e+08,8.000000e+05,2.090386e+08,...,4.744467e+08,0.0,2.018711e+08,1.615997e+08,0.0,110500,CAJA,2022-12-01,2022,12
3,2.791246e+08,1.061263e+07,0.0,8.180852e+08,0.000000e+00,3.589941e+08,1.718631e+08,1.146549e+08,0.000000e+00,2.080386e+08,...,4.738467e+08,0.0,1.998711e+08,1.600997e+08,0.0,110505,CAJA GENERAL,2022-12-01,2022,12
4,7.680000e+06,0.000000e+00,0.0,4.850000e+06,1.100000e+06,0.000000e+00,0.000000e+00,9.500000e+06,8.000000e+05,1.000000e+06,...,6.000000e+05,0.0,2.000000e+06,1.500000e+06,0.0,110510,CAJA MENOR,2022-12-01,2022,12


Generate new columns for the DataFrame

In [9]:
keys = list(coops.columns)
keys.remove("CUENTA")
keys.remove('NOMBRE CUENTA')
keys.remove('DATE')
keys.remove('YEAR')
keys.remove('MONTH')

indexes_camel = ['Quebranto Patrimonial', 'Relación Solvencia', 'Relación entre Aportes sociales mínimos no reducibles y Capital Social', 'Relación entre el Capital Institucional y el Activo Total', 
                 'Indicador de calidad por riesgo', 'Indicador de calidad por riesgo con castigos', 'Indicador de Cobertura de la Cartera Total en Riesgo', 'Activo Productivo', 'Indicador de Cobertura individual de la cartera improductiva para la cartera en Riesgo',
                 'Indicador de Margen Financiero de Operación', 'Indicador de Margen Operacional', 'Indicador de relación entre las obligaciones financieras y el pasivo total', 'Estructura de Balance',
                 'Indicador de rentabilidad sobre recursos propios - ROE', 'Indicador de margen neto', 'Indicador de rentabilidad sobre el capital invertido - ROIC',
                 'Indicador de Riesgo de Liquidez - IRL']
years = range(2022, 2026)
months = range(1, 13)

We need to parallelize the next cells, because there is a lot of calculations there (for the big dataset)

In [10]:
from concurrent.futures import ThreadPoolExecutor
from functools import partial

### Capital

In [15]:
def calculate_values_capital(company, coops, years, months, indices_camel):
    # Obtener TODOS los datos de la cooperativa una sola vez
    cols = ['CUENTA', 'NOMBRE CUENTA', 'YEAR', "MONTH", company]
    company_data = coops[cols].copy()

    rows = []
    for y in years:
        for m in months:
            data = company_data[
                (company_data["YEAR"] == y) &
                (company_data["MONTH"] == m)
            ]
            
            c1 = loss_of_assets(data, company, y, m)
            c2 = solvency_ratio()
            c3 = min_social_contrib_social_capital(data, company, y, m)
            c4 = capital_contribution_ratio(data, company, y, m)

            rows.append([y, m, indices_camel[0], company, c1])
            rows.append([y, m, indices_camel[1], company, c2])
            rows.append([y, m, indices_camel[2], company, c3])
            rows.append([y, m, indices_camel[3], company, c4])
    return rows

# parameters
partial_func_c = partial(calculate_values_capital, coops=coops, years=years, months=months, indices_camel=indexes_camel)

# execute for every company
with ThreadPoolExecutor() as executor:
    resultados = list(executor.map(partial_func_c, keys))

# organize all the info in a dataframe
rows_c = [fila for sublista in resultados for fila in sublista]

df = pd.DataFrame(rows_c, columns=['Año', 'Mes', 'índice CAMEL', 'Cooperativa', 'valor CAMEL'])
df_pivot = df.pivot_table(index=['Año', 'Mes', 'índice CAMEL'], columns='Cooperativa', values='valor CAMEL').reset_index()
df_pivot.columns.name = None

In [16]:
df_pivot.to_csv("../../tablas/camel/Capital.csv", index=False, encoding="utf-8")

### Assets

In [12]:
def calculate_values_assets(company, coops, years, months, indices_camel):
    cols = ['CUENTA', 'NOMBRE CUENTA', 'YEAR', "MONTH", company]
    company_data = coops[cols].copy()
    rows = []
    for y in years:
        for m in months:
            data = company_data[
                            (company_data["YEAR"] == y) &
                            (company_data["MONTH"] == m)
                        ]
            a1 = risk_quality_indicator(data, company, y, m)
            a2 = risk_quality_with_writeoffs(data, company, y, m)


            rows.append([y, m, indices_camel[4], company, a1])
            rows.append([y, m, indices_camel[5], company, a2])

    return rows

partial_func_a = partial(calculate_values_assets, coops=coops, years=years, months=months, indices_camel=indexes_camel)

# every company
with ThreadPoolExecutor() as executor:
    resultados = list(executor.map(partial_func_a, keys))

rows_a = [fila for sublista in resultados for fila in sublista]

df_a = pd.DataFrame(rows_a, columns=['Año', 'Mes', 'índice CAMEL', 'Cooperativa', 'valor CAMEL'])
df_pivot_a = df_a.pivot_table(index=['Año', 'Mes', 'índice CAMEL'], columns='Cooperativa', values='valor CAMEL').reset_index()
df_pivot_a.columns.name = None

In [14]:
def calculate_values_assets2(company, coops, years, months, indices_camel):
    cols = ['CUENTA', 'NOMBRE CUENTA', 'YEAR', "MONTH", company]
    company_data = coops[cols].copy()
    rows = []
    for y in years:
        for m in months:
            data = company_data[
                            (company_data["YEAR"] == y) &
                            (company_data["MONTH"] == m)
                        ]
            a3 = total_risk_coverage_indicator(data, company, y, m)
            a4 = productive_assets_ratio(data, company, y, m)
            a5 = individual_coverage_nonproductive_portfolio(data, company, y, m)

            rows.append([y, m, indices_camel[6], company, a3])
            rows.append([y, m, indices_camel[7], company, a4])
            rows.append([y, m, indices_camel[8], company, a5])
    return rows

partial_func_a2 = partial(calculate_values_assets2, coops=coops, years=years, months=months, indices_camel=indexes_camel)

# every company
with ThreadPoolExecutor() as executor:
    resultados = list(executor.map(partial_func_a2, keys))

rows_a2 = [fila for sublista in resultados for fila in sublista]

df_a2 = pd.DataFrame(rows_a2, columns=['Año', 'Mes', 'índice CAMEL', 'Cooperativa', 'valor CAMEL'])
df_pivot_a2 = df_a2.pivot_table(index=['Año', 'Mes', 'índice CAMEL'], columns='Cooperativa', values='valor CAMEL').reset_index()
df_pivot_a2.columns.name = None

In [15]:
assets = pd.concat([df_pivot_a, df_pivot_a2], ignore_index=True)
assets.to_csv("../../tablas/camel/Assets.csv", index=False, encoding="utf-8")

### Managerial

In [16]:
def calculate_values_managerial(company, coops, years, months, indices_camel):
    cols = ['CUENTA', 'NOMBRE CUENTA', 'YEAR', "MONTH", company]
    company_data = coops[cols].copy()
    rows = []
    for y in years:
        for m in months:
            data = company_data[
                            (company_data["YEAR"] == y) &
                            (company_data["MONTH"] == m)
                        ]
            # values for M: MANAGERIAL
            m1 = financial_margin_operation(data, company, y, m)
            m2 = operational_margin(data, company, y, m)
            m3 = financial_obligations_ratio(data, company, y, m)
            m4 = balance_structure(data, company, y, m)

            rows.append([y, m, indices_camel[9], company, m1])
            rows.append([y, m, indices_camel[10], company, m2])
            rows.append([y, m, indices_camel[11], company, m3])
            rows.append([y, m, indices_camel[12], company, m4])
    return rows


In [17]:
partial_func_m = partial(calculate_values_managerial, coops=coops, years=years, months=months, indices_camel=indexes_camel)

# every company
with ThreadPoolExecutor() as executor:
    resultados = list(executor.map(partial_func_m, keys))

rows_m = [fila for sublista in resultados for fila in sublista]

df_m = pd.DataFrame(rows_m, columns=['Año', 'Mes', 'índice CAMEL', 'Cooperativa', 'valor CAMEL'])
df_pivot_m = df_m.pivot_table(index=['Año', 'Mes', 'índice CAMEL'], columns='Cooperativa', values='valor CAMEL').reset_index()
df_pivot_m.columns.name = None

In [18]:
df_pivot_m.to_csv("../../tablas/camel/Managerial.csv", index=False, encoding="utf-8")

### Earnings

In [11]:
def calculate_values_earnings(company, coops, years, months, indices_camel):
    cols = ['CUENTA', 'NOMBRE CUENTA', 'YEAR', "MONTH", company]
    data = coops[cols].copy()
    rows = []
    for y in years:
        for m in months:
            # values for E: EARNINGS
            e1 = ROE(data, company, y, m)
            e2 = net_margin_indicator(data, company, y, m)
            e3 = ROIC(data, company, y, m)

            rows.append([y, m, indices_camel[13], company, e1])
            rows.append([y, m, indices_camel[14], company, e2])
            rows.append([y, m, indices_camel[15], company, e3])
    return rows

In [14]:
partial_func_e = partial(calculate_values_earnings, coops=coops, years=years, months=months, indices_camel=indexes_camel)

# every company
with ThreadPoolExecutor() as executor:
    resultados = list(executor.map(partial_func_e, keys))

rows_e = [fila for sublista in resultados for fila in sublista]

df_e = pd.DataFrame(rows_e, columns=['Año', 'Mes', 'índice CAMEL', 'Cooperativa', 'valor CAMEL'])
df_pivot_e = df_e.pivot_table(index=['Año', 'Mes', 'índice CAMEL'], columns='Cooperativa', values='valor CAMEL').reset_index()
df_pivot_e.columns.name = None

In [15]:
df_pivot_e.to_csv("../../tablas/camel/Earnings.csv", index=False, encoding="utf-8")

### Liquidicy

In [22]:
def calculate_values_liquidicy(company, coops, years, months, indices_camel):
    cols = ['CUENTA', 'NOMBRE CUENTA', 'YEAR', "MONTH", company]
    company_data = coops[cols].copy()
    rows = []
    for y in years:
        for m in months:
            data = company_data[
                            (company_data["YEAR"] == y) &
                            (company_data["MONTH"] == m)
                        ]
            # values for L: LIQUIDICY
            l1 = liquidity_risk(data, company, y, m)
            rows.append([y, m, indices_camel[16], company, l1])
    return rows

In [23]:
partial_func_l = partial(calculate_values_liquidicy, coops=coops, years=years, months=months, indices_camel=indexes_camel)

# every company
with ThreadPoolExecutor() as executor:
    resultados = list(executor.map(partial_func_l, keys))

rows_l = [fila for sublista in resultados for fila in sublista]

df_l = pd.DataFrame(rows_l, columns=['Año', 'Mes', 'índice CAMEL', 'Cooperativa', 'valor CAMEL'])
df_pivot_l = df_l.pivot_table(index=['Año', 'Mes', 'índice CAMEL'], columns='Cooperativa', values='valor CAMEL').reset_index()
df_pivot_l.columns.name = None

In [24]:
df_pivot_l.to_csv("../../tablas/camel/Liquidity.csv", index=False, encoding="utf-8")

## CSV for BD

Write on .csv the data from DataFrames:

DataFrames's names and their title:
* 'df_pivot': Capital
* 'df_pivot_a' y 'df_pivot_a_2': Assets
* 'df_pivot_m': Managerial
* 'df_pivot_e': Earnings
* 'df_pivot_l': Liquidicy